<a href="https://colab.research.google.com/github/mahmoudmayaleh/BERT-Sentiment-Augmentation/blob/main/V2_Enhancing_Sentiment_Classification_on_Small_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### _Enhancing Sentiment Classification on Small Datasets through Data Augmentation and Transfer Learning: A Comparative Study_

##Notebook Description:

__The Objective:__ This notebook supports the study _"Enhancing Sentiment Classification on Small Datasets through Data Augmentation and Transfer Learning"_. It explores the impact of data augmentation techniques: __Easy Data Augmentation (EDA)__, __NLPaug__ and __Back-Translation__ on sentiment classification performance. Classical machine learning models, including __Logistic Regression__ and __Random Forest__, are trained on both the original and augmented datasets. Additionally, a __BERT__ model is fine-tuned on the original dataset and again on a dataset augmented using NLPaug, leveraging transfer learning. Model performance is evaluated using metrics such as Accuracy, F1-score, and AUC.

## Authors:
### _Mahmoud Mayaleh_ & _Samer Mayaleh_

## Step 1: Installing dependencies

In [ ]:
!pip install scikit-learn pandas torch transformers
!pip install -q nlpaug
!pip install -q sentence-transformers
!pip install --upgrade datasets
!pip install -U httpx==0.27.0 datasets==3.1.0

__Importing Librarys:__

In [ ]:
import os
import random
import re
import time
import zipfile
from io import StringIO

import httpx
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.stats import t
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.utils import shuffle
from torch import amp, cuda
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import BertForSequenceClassification, BertTokenizer, pipeline

import nlpaug.augmenter.word as naw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## Step 2: Dataset preparation

__Importing Data:__

In [ ]:
dataset = load_dataset("imdb", split="train")

In [ ]:
df = pd.DataFrame(dataset)

In [ ]:
# Sample 5000 entries to simulate a small dataset
df_small = df.sample(n=5000, random_state=42)[['text', 'label']]
df_small = df_small.rename(columns={'text': 'review'})

In [ ]:
#shuffling data
df_small = shuffle(df_small, random_state=42).reset_index(drop=True)

In [ ]:
vectorizer = TfidfVectorizer()
X_vect = vectorizer.fit_transform(df_small['review'])
y = df_small['label'].values

## Step 3: Train Classical ML Models (Baseline)

Stratified K-Fold

In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
accuracies = []
f1_scores = []
train_times = []

Logistic Regression

In [ ]:
for train_idx, test_idx in kfold.split(X_vect, y):
    X_train, X_test = X_vect[train_idx], X_vect[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LogisticRegression()

    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()

    train_duration = end_time - start_time
    train_times.append(train_duration)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    accuracies.append(acc)
    f1_scores.append(f1)

    print(f"Fold done. Accuracy: {acc:.4f}, F1: {f1:.4f}, Train Time: {train_duration:.2f} sec")

In [ ]:
# Confidence Intervals
def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)
    h = sem * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean, h

mean_acc, ci_acc = confidence_interval(accuracies)
mean_f1, ci_f1 = confidence_interval(f1_scores)
print("Logistic Regression (Original dataset)")
print(f"Accuracy: {mean_acc:.4f} ± {ci_acc:.4f}")
print(f"F1 Score: {mean_f1:.4f} ± {ci_f1:.4f}")

In [ ]:
for train_idx, test_idx in kfold.split(X_vect, y):
    X_train, X_test = X_vect[train_idx], X_vect[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = RandomForestClassifier(n_estimators=100, random_state=42)

    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()

    train_duration = end_time - start_time
    train_times.append(train_duration)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    accuracies.append(acc)
    f1_scores.append(f1)

    print(f"Fold done. Accuracy: {acc:.4f}, F1: {f1:.4f}, Train Time: {train_duration:.2f} sec")

In [ ]:
# Confidence Intervals
def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)
    h = sem * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean, h

mean_acc, ci_acc = confidence_interval(accuracies)
mean_f1, ci_f1 = confidence_interval(f1_scores)
print("Random Forest (Original dataset)")
print(f"Accuracy: {mean_acc:.4f} ± {ci_acc:.4f}")
print(f"F1 Score: {mean_f1:.4f} ± {ci_f1:.4f}")

## Step 4: Data Augmentation for Small Dataset

In [ ]:
#Easy Data Augmentation (EDA): (Random Insertion + Random Swap + Random Deletion)

def random_deletion(words, p=0.1):
    if len(words) == 1:
        return words
    return [w for w in words if random.random() > p]

def random_swap(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(new_words)), 2)
        new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return new_words

def random_insertion(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        idx = random.randint(0, len(new_words)-1)
        new_words.insert(idx, new_words[idx])
    return new_words

def eda(sentence, alpha_rd=0.1, alpha_rs=1, alpha_ri=1):
    words = sentence.split()
    if len(words) < 4: return sentence
    augmented = random_deletion(words, p=alpha_rd)
    augmented = random_swap(augmented, n=alpha_rs)
    augmented = random_insertion(augmented, n=alpha_ri)
    return ' '.join(augmented)
#--------------------------------------------

df_eda = pd.DataFrame({
    'review': df_small['review'].apply(lambda x: eda(x)),
    'label': df_small['label']
})

-----------------------

In [ ]:
#Random Insertion/Swap/Deletion using nlpaug
aug = naw.RandomWordAug(action="swap")
df_nlpaug = pd.DataFrame({
    'review': df_small['review'].apply(lambda x: aug.augment(x)),
    'label': df_small['label']
})


---------------------------------------

In [ ]:
# Path to saved back-translation CSV
bt_dir = "/content/drive/MyDrive/bert_datasets"
bt_path = os.path.join(bt_dir, "imdb_backtranslation_5k_en_fr_en.csv")
if os.path.exists(bt_path):
    print(f"Loading back-translated dataset from {bt_path}")
    df_backtranslation = pd.read_csv(bt_path)
else:
    print("Back-translation CSV not found, generating from scratch...")

    device_arg = 0 if torch.cuda.is_available() else -1

    en_fr_translator = pipeline(
        "translation_en_to_fr",
        model="Helsinki-NLP/opus-mt-en-fr",
        device=device_arg,
    )
    fr_en_translator = pipeline(
        "translation_fr_to_en",
        model="Helsinki-NLP/opus-mt-fr-en",
        device=device_arg,
    )

    def back_translate(text):
        try:
            if not isinstance(text, str) or not text.strip():
                return text
            fr = en_fr_translator(text, max_length=512)[0]["translation_text"]
            en_bt = fr_en_translator(fr, max_length=512)[0]["translation_text"]
            return en_bt
        except Exception:
            return text

    batch = df_small.copy().reset_index(drop=True)
    tqdm.pandas()
    batch["backtranslated_review"] = batch["review"].progress_apply(back_translate)

    df_backtranslation = pd.DataFrame({
        "review": batch["backtranslated_review"],
        "label": batch["label"],
    })

    os.makedirs(bt_dir, exist_ok=True)
    df_backtranslation.to_csv(bt_path, index=False)
    print(f"Saved back-translated dataset to {bt_path}")


--------------------------

### Similarity analysis (Original vs augmented)

In [ ]:
model_sbert = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device='cpu'
)# Use up to 1000 pairs
N = min(1000, len(df_small))
orig_texts = df_small['review'].astype(str).tolist()[:N]
eda_texts = df_eda['review'].astype(str).tolist()[:N]
nlpaug_texts = df_nlpaug['review'].astype(str).tolist()[:N]

# For back-translation, align with its own subset
M = len(df_backtranslation)
bt_orig_texts = df_small['review'].astype(str).tolist()[:M]
bt_texts = df_backtranslation['review'].astype(str).tolist()

emb_orig = model_sbert.encode(orig_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
emb_eda = model_sbert.encode(eda_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
emb_nlpaug = model_sbert.encode(nlpaug_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
emb_bt_orig = model_sbert.encode(bt_orig_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
emb_bt = model_sbert.encode(bt_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

def pairwise_diag_cosine(a, b):
    sims = np.sum(a * b, axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return sims

sims_eda = pairwise_diag_cosine(emb_orig, emb_eda)
sims_nlpaug = pairwise_diag_cosine(emb_orig, emb_nlpaug)
sims_bt = pairwise_diag_cosine(emb_bt_orig, emb_bt)

print("EDA mean similarity:", float(sims_eda.mean()), "std:", float(sims_eda.std()))
print("NLPaug mean similarity:", float(sims_nlpaug.mean()), "std:", float(sims_nlpaug.std()))
print("Back-translation mean similarity:", float(sims_bt.mean()), "std:", float(sims_bt.std()))


In [ ]:
eda_mean, eda_std = float(sims_eda.mean()), float(sims_eda.std())
nlpaug_mean, nlpaug_std = float(sims_nlpaug.mean()), float(sims_nlpaug.std())
bt_mean, bt_std = float(sims_bt.mean()), float(sims_bt.std())

print("EDA similarity:    mean =", eda_mean, "std =", eda_std)
print("NLPaug similarity: mean =", nlpaug_mean, "std =", nlpaug_std)
print("BT similarity:     mean =", bt_mean, "std =", bt_std)


In [ ]:
def show_examples(n=7, seed=77):
    np.random.seed(seed)
    idxs = np.random.choice(len(df_small), size=n, replace=False)
    for i in idxs:
        orig = df_small.iloc[i]["review"]
        eda  = df_eda.iloc[i]["review"]
        nlp  = df_nlpaug.iloc[i]["review"]
        bt   = df_backtranslation.iloc[i]["review"]
        print("Example", i)
        print("Original:       ", orig)
        print("EDA:            ", eda)
        print("NLPaug:         ", nlp)
        print("Back-translation:", bt)
        print("-" * 80)

show_examples(7)

##  Step 5: Retrain using the original and augmented text

In [ ]:
def confidence_interval(data, confidence=0.95):
    data = np.array(data)
    n = len(data)
    mean = np.mean(data)
    sem = np.std(data, ddof=1) / np.sqrt(n)
    ci_bounds = t.interval(confidence, n - 1, loc=mean, scale=sem)
    return mean, ci_bounds[0], ci_bounds[1]

def evaluate_and_store_metrics_kfold(df, model, model_name, augmentation_name, k=10):
    X = [' '.join(text) if isinstance(text, list) else str(text) for text in df['review']]
    y = df['label']

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    accuracies, precisions, recalls, f1s, aucs = [], [], [], [], []
    total_time = 0

    for train_index, test_index in skf.split(X, y):
        X_train = [X[i] for i in train_index]
        X_test = [X[i] for i in test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]

        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
            ('clf', model)
        ])

        start = time.time()
        pipeline.fit(X_train, y_train)
        total_time += time.time() - start

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline.named_steps['clf'], 'predict_proba') else None

        accuracies.append(accuracy_score(y_test, y_pred))
        precisions.append(precision_score(y_test, y_pred))
        recalls.append(recall_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred))
        if y_prob is not None:
            aucs.append(roc_auc_score(y_test, y_prob))

    acc_mean, acc_low, acc_high = confidence_interval(accuracies)
    prec_mean, prec_low, prec_high = confidence_interval(precisions)
    rec_mean, rec_low, rec_high = confidence_interval(recalls)
    f1_mean, f1_low, f1_high = confidence_interval(f1s)
    auc_mean, auc_low, auc_high = confidence_interval(aucs) if aucs else (None, None, None)

    return {
        "Augmentation": augmentation_name,
        "Model": model_name,
        "Accuracy": acc_mean,
        "Accuracy CI Lower": acc_low,
        "Accuracy CI Upper": acc_high,
        "Precision": prec_mean,
        "Precision CI Lower": prec_low,
        "Precision CI Upper": prec_high,
        "Recall": rec_mean,
        "Recall CI Lower": rec_low,
        "Recall CI Upper": rec_high,
        "F1": f1_mean,
        "F1 CI Lower": f1_low,
        "F1 CI Upper": f1_high,
        "AUC": auc_mean,
        "AUC CI Lower": auc_low,
        "AUC CI Upper": auc_high,
        "Accuracy Folds": accuracies,
        "F1 Folds": f1s,
        "Training Time (s)": round(total_time, 2)
    }

In [ ]:
results = []

augmentations = [
    (df_small, "Original"),
    (df_eda, "EDA"),
    (df_nlpaug, "NLPaug"),
    (df_backtranslation, "Back-translation")
]

models = [
    (LogisticRegression(max_iter=1000), "Logistic Regression"),
    (RandomForestClassifier(n_estimators=100), "Random Forest")
]

for df_aug, aug_name in augmentations:
    for model, model_name in models:
        results.append(evaluate_and_store_metrics_kfold(df_aug, model, model_name, aug_name))

        # Original + Augmented
        if aug_name != "Original":
            df_combined = pd.concat([df_small, df_aug], ignore_index=True)
            results.append(evaluate_and_store_metrics_kfold(df_combined, model, model_name, f"Original + {aug_name}"))

results_df = pd.DataFrame(results).round(4)
results_df


In [ ]:
from scipy.stats import ttest_rel

def cohen_d(x, y):
    x = np.array(x)
    y = np.array(y)
    diff = y - x
    return diff.mean() / diff.std(ddof=1)

effect_rows = []

for model_name in ["Logistic Regression", "Random Forest"]:
    # Baseline folds: Original
    base_row = next(
        r for r in results
        if r["Model"] == model_name and r["Augmentation"] == "Original"
    )
    base_acc_folds = base_row["Accuracy Folds"]

    for aug_name in ["Original + EDA", "Original + NLPaug", "Original + Back-translation"]:
        aug_row = next(
            (r for r in results if r["Model"] == model_name and r["Augmentation"] == aug_name),
            None,
        )
        if aug_row is None:
            continue

        aug_acc_folds = aug_row["Accuracy Folds"]

        # Paired t-test (per fold)
        t_stat, p_val = ttest_rel(base_acc_folds, aug_acc_folds)
        d = cohen_d(base_acc_folds, aug_acc_folds)

        effect_rows.append({
            "Model": model_name,
            "Augmentation": aug_name,
            "Baseline Acc Mean": np.mean(base_acc_folds),
            "Augmented Acc Mean": np.mean(aug_acc_folds),
            "Cohen_d": d,
            "p_value": p_val,
        })

effect_df = pd.DataFrame(effect_rows).round(4)
print("Effect sizes and significance (Accuracy, 10-fold, paired):")
print(effect_df)


In [ ]:
results_df.to_csv("/content/drive/MyDrive/bert_models/classical_results_df.csv", index=False)
effect_df.to_csv("/content/drive/MyDrive/bert_models/classical_effect_df.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Combined dataset: Original + NLPaug
df_combined = pd.concat([df_small, df_nlpaug], ignore_index=True)
X = df_combined['review'].astype(str).tolist()
y = df_combined['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
cm_lr = confusion_matrix(y_test, y_pred_lr)
print("Confusion matrix – Logistic Regression (Original + NLPaug):")
print(cm_lr)

# Random Forest
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion matrix – Random Forest (Original + NLPaug):")
print(cm_rf)


## Step 6: Transfer Learning - __BERT__

In [ ]:
device = torch.device("cuda" if cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

seed_list = [12, 44, 127]
num_epochs = 10

# Prepare ORIGINAL dataset tensors
orig_texts = df_small['review'].astype(str).tolist()
orig_labels = df_small['label'].values

encoded_orig = tokenizer(orig_texts, padding=True, truncation=True, max_length=256, return_tensors='pt')
input_ids_orig = encoded_orig['input_ids']
attention_mask_orig = encoded_orig['attention_mask']
labels_tensor_orig = torch.tensor(orig_labels)

# Prepare ORIGINAL + NLPaug dataset tensors
df_aug_all = pd.concat([df_small, df_nlpaug], ignore_index=True)
texts_aug = df_aug_all['review'].astype(str).tolist()
labels_aug = df_aug_all['label'].values

encoded_aug = tokenizer(texts_aug, padding=True, truncation=True, max_length=256, return_tensors='pt')
input_ids_aug = encoded_aug['input_ids']
attention_mask_aug = encoded_aug['attention_mask']
labels_tensor_aug = torch.tensor(labels_aug)

def create_loader_from_tensors(input_ids, attention_mask, labels_tensor, indices, batch_size=16, shuffle=True):
    dataset = TensorDataset(input_ids[indices], attention_mask[indices], labels_tensor[indices])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)

results_orig = {}
results_aug = {}

In [ ]:
print("=== Training BERT on ORIGINAL dataset for seeds", seed_list, "===")

results_orig = {}
save_dir_orig = "/content/drive/MyDrive/bert_models/original"
os.makedirs(save_dir_orig, exist_ok=True)

for SEED in seed_list:
    print(f"\n--- BERT ORIGINAL, SEED {SEED} ---")
    set_seed(SEED)

    # Train/Val/Test split for ORIGINAL
    train_idx_o, temp_idx_o = train_test_split(
        np.arange(len(orig_labels)),
        test_size=0.2,
        stratify=orig_labels,
        random_state=SEED,
    )
    val_idx_o, test_idx_o = train_test_split(
        temp_idx_o,
        test_size=0.5,
        stratify=orig_labels[temp_idx_o],
        random_state=SEED,
    )

    train_loader_orig = create_loader_from_tensors(
        input_ids_orig, attention_mask_orig, labels_tensor_orig, train_idx_o
    )
    val_loader_orig = create_loader_from_tensors(
        input_ids_orig, attention_mask_orig, labels_tensor_orig, val_idx_o, shuffle=False
    )
    test_loader_orig = create_loader_from_tensors(
        input_ids_orig, attention_mask_orig, labels_tensor_orig, test_idx_o, shuffle=False
    )

    model_orig = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2
    ).to(device)
    optimizer_orig = AdamW(model_orig.parameters(), lr=2e-5)
    scheduler_orig = StepLR(optimizer_orig, step_size=1, gamma=0.9)
    loss_fn_orig = CrossEntropyLoss()
    scaler_orig = amp.GradScaler(device=device.type)

    train_losses_orig = []
    val_losses_orig = []
    acc_epochs_orig = []
    auc_epochs_orig = []

    best_val_auc = -1.0
    best_state_dict = None
    best_epoch = 0

    start_time = time.time()
    for epoch in range(num_epochs):
        model_orig.train()
        epoch_train_loss = 0.0
        train_count = 0

        for batch in train_loader_orig:
            input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
            optimizer_orig.zero_grad()

            with amp.autocast(device_type=device.type):
                outputs = model_orig(
                    input_ids=input_ids_b,
                    attention_mask=attention_mask_b,
                    labels=labels_b,
                )
                loss = outputs.loss

            epoch_train_loss += loss.item() * labels_b.size(0)
            train_count += labels_b.size(0)

            scaler_orig.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model_orig.parameters(), max_norm=1.0)
            scaler_orig.step(optimizer_orig)
            scaler_orig.update()

        scheduler_orig.step()
        train_losses_orig.append(epoch_train_loss / train_count)

        # Validation
        model_orig.eval()
        val_loss, correct, total = 0.0, 0, 0
        all_probs, all_labels_eval = [], []

        with torch.no_grad():
            for batch in val_loader_orig:
                input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
                with amp.autocast(device_type=device.type):
                    outputs = model_orig(
                        input_ids=input_ids_b,
                        attention_mask=attention_mask_b,
                    )
                    logits = outputs.logits
                    loss_v = loss_fn_orig(logits.float(), labels_b)
                val_loss += loss_v.item() * labels_b.size(0)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels_b).sum().item()
                total += labels_b.size(0)
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels_eval.extend(labels_b.cpu().numpy())

        avg_val_loss = val_loss / total
        val_losses_orig.append(avg_val_loss)
        val_acc = correct / total
        acc_epochs_orig.append(val_acc)
        try:
            val_auc = roc_auc_score(all_labels_eval, all_probs)
        except:
            val_auc = float("nan")
        auc_epochs_orig.append(val_auc)

        print(
            f"[Original SEED {SEED}] Epoch {epoch+1:02d} | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}"
        )

        # Early stopping tracking (best by Val AUC)
        if not np.isnan(val_auc) and val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch + 1
            best_state_dict = model_orig.state_dict()

    end_time = time.time()
    train_time_minutes = (end_time - start_time) / 60.0
    print(f"[Original SEED {SEED}] Total training time: {train_time_minutes:.2f} minutes")
    print(f"[Original SEED {SEED}] Best epoch by Val AUC: {best_epoch} (AUC={best_val_auc:.4f})")

    # Load best checkpoint before test
    if best_state_dict is not None:
        model_orig.load_state_dict(best_state_dict)

    # Evaluate on test set
    model_orig.eval()
    all_preds_o, all_labels_o, all_probs_o = [], [], []
    with torch.no_grad():
        for batch in test_loader_orig:
            input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
            outputs = model_orig(
                input_ids=input_ids_b,
                attention_mask=attention_mask_b,
            )
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_preds_o.extend(preds)
            all_probs_o.extend(probs)
            all_labels_o.extend(labels_b.cpu().numpy())

    acc_test = accuracy_score(all_labels_o, all_preds_o)
    f1_test = f1_score(all_labels_o, all_preds_o)
    auc_test = roc_auc_score(all_labels_o, all_probs_o)
    cm_orig = confusion_matrix(all_labels_o, all_preds_o)

    print(
        f"[Original SEED {SEED}] Test Accuracy: {acc_test:.4f}, "
        f"F1: {f1_test:.4f}, AUC: {auc_test:.4f}"
    )
    print("Confusion matrix – BERT (Original):")
    print(cm_orig)

    # Save best weights for this seed
    torch.save(
        model_orig.state_dict(),
        os.path.join(save_dir_orig, f"bert_original_seed{SEED}.pt"),
    )

    results_orig[SEED] = {
        "train_losses": train_losses_orig,
        "val_losses": val_losses_orig,
        "acc_epochs": acc_epochs_orig,
        "auc_epochs": auc_epochs_orig,
        "test_acc": acc_test,
        "test_f1": f1_test,
        "test_auc": auc_test,
        "cm": cm_orig,
        "train_time_min": train_time_minutes,
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
    }

In [ ]:
print("\n=== Training BERT on ORIGINAL + NLPaug dataset for seeds", seed_list, "===")

results_aug = {}

save_dir_aug = "/content/drive/MyDrive/bert_models/orig_nlpaug"
os.makedirs(save_dir_aug, exist_ok=True)

for SEED in seed_list:
    print(f"\n--- BERT ORIGINAL+NLPaug, SEED {SEED} ---")
    set_seed(SEED)

    # Train/Val/Test split for ORIGINAL+NLPaug
    train_idx_a, temp_idx_a = train_test_split(
        np.arange(len(labels_aug)),
        test_size=0.2,
        stratify=labels_aug,
        random_state=SEED,
    )
    val_idx_a, test_idx_a = train_test_split(
        temp_idx_a,
        test_size=0.5,
        stratify=labels_aug[temp_idx_a],
        random_state=SEED,
    )

    train_loader_aug = create_loader_from_tensors(
        input_ids_aug, attention_mask_aug, labels_tensor_aug, train_idx_a
    )
    val_loader_aug = create_loader_from_tensors(
        input_ids_aug, attention_mask_aug, labels_tensor_aug, val_idx_a, shuffle=False
    )
    test_loader_aug = create_loader_from_tensors(
        input_ids_aug, attention_mask_aug, labels_tensor_aug, test_idx_a, shuffle=False
    )

    model_aug = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2
    ).to(device)
    optimizer_aug = AdamW(model_aug.parameters(), lr=2e-5)
    scheduler_aug = StepLR(optimizer_aug, step_size=1, gamma=0.9)
    loss_fn_aug = CrossEntropyLoss()
    scaler_aug = amp.GradScaler(device=device.type)

    train_losses_aug = []
    val_losses_aug = []
    acc_epochs_aug = []
    auc_epochs_aug = []

    best_val_auc = -1.0
    best_state_dict = None
    best_epoch = 0

    start_time = time.time()
    for epoch in range(num_epochs):
        model_aug.train()
        epoch_train_loss = 0.0
        train_count = 0

        for batch in train_loader_aug:
            input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
            optimizer_aug.zero_grad()

            with amp.autocast(device_type=device.type):
                outputs = model_aug(
                    input_ids=input_ids_b,
                    attention_mask=attention_mask_b,
                    labels=labels_b,
                )
                loss = outputs.loss

            epoch_train_loss += loss.item() * labels_b.size(0)
            train_count += labels_b.size(0)

            scaler_aug.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model_aug.parameters(), max_norm=1.0)
            scaler_aug.step(optimizer_aug)
            scaler_aug.update()

        scheduler_aug.step()
        train_losses_aug.append(epoch_train_loss / train_count)

        # Validation
        model_aug.eval()
        val_loss, correct, total = 0.0, 0, 0
        all_probs, all_labels_eval = [], []

        with torch.no_grad():
            for batch in val_loader_aug:
                input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
                with amp.autocast(device_type=device.type):
                    outputs = model_aug(
                        input_ids=input_ids_b,
                        attention_mask=attention_mask_b,
                    )
                    logits = outputs.logits
                    loss_v = loss_fn_aug(logits.float(), labels_b)
                val_loss += loss_v.item() * labels_b.size(0)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels_b).sum().item()
                total += labels_b.size(0)
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels_eval.extend(labels_b.cpu().numpy())

        avg_val_loss = val_loss / total
        val_losses_aug.append(avg_val_loss)
        val_acc = correct / total
        acc_epochs_aug.append(val_acc)
        try:
            val_auc = roc_auc_score(all_labels_eval, all_probs)
        except:
            val_auc = float("nan")
        auc_epochs_aug.append(val_auc)

        print(
            f"[Orig+NLPaug SEED {SEED}] Epoch {epoch+1:02d} | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}"
        )

        # Early stopping tracking (best by Val AUC)
        if not np.isnan(val_auc) and val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch + 1
            best_state_dict = model_aug.state_dict()

    end_time = time.time()
    train_time_minutes = (end_time - start_time) / 60.0
    print(f"[Orig+NLPaug SEED {SEED}] Total training time: {train_time_minutes:.2f} minutes")
    print(f"[Orig+NLPaug SEED {SEED}] Best epoch by Val AUC: {best_epoch} (AUC={best_val_auc:.4f})")

    # Load best checkpoint before test
    if best_state_dict is not None:
        model_aug.load_state_dict(best_state_dict)

    # Evaluate on test set
    model_aug.eval()
    all_preds_a, all_labels_a, all_probs_a = [], [], []
    with torch.no_grad():
        for batch in test_loader_aug:
            input_ids_b, attention_mask_b, labels_b = [b.to(device) for b in batch]
            outputs = model_aug(
                input_ids=input_ids_b,
                attention_mask=attention_mask_b,
            )
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_preds_a.extend(preds)
            all_probs_a.extend(probs)
            all_labels_a.extend(labels_b.cpu().numpy())

    acc_test = accuracy_score(all_labels_a, all_preds_a)
    f1_test = f1_score(all_labels_a, all_preds_a)
    auc_test = roc_auc_score(all_labels_a, all_probs_a)
    cm_aug = confusion_matrix(all_labels_a, all_preds_a)

    print(
        f"[Orig+NLPaug SEED {SEED}] Test Accuracy: {acc_test:.4f}, "
        f"F1: {f1_test:.4f}, AUC: {auc_test:.4f}"
    )
    print("Confusion matrix – BERT (Original + NLPaug):")
    print(cm_aug)

    # Save best weights for this seed
    torch.save(
        model_aug.state_dict(),
        os.path.join(save_dir_aug, f"bert_orig_nlpaug_seed{SEED}.pt"),
    )

    results_aug[SEED] = {
        "train_losses": train_losses_aug,
        "val_losses": val_losses_aug,
        "acc_epochs": acc_epochs_aug,
        "auc_epochs": auc_epochs_aug,
        "test_acc": acc_test,
        "test_f1": f1_test,
        "test_auc": auc_test,
        "cm": cm_aug,
        "train_time_min": train_time_minutes,
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
    }

In [ ]:
import pickle, os
os.makedirs("/content/drive/MyDrive/bert_results", exist_ok=True)

with open("/content/drive/MyDrive/bert_results/results_orig.pkl", "wb") as f:
    pickle.dump(results_orig, f)

with open("/content/drive/MyDrive/bert_results/results_aug.pkl", "wb") as f:
    pickle.dump(results_aug, f)

## Step 7: Evaluation & Plotting

In [ ]:
summary_rows = []
for SEED, m in results_orig.items():
    summary_rows.append({
        "Setting": "Original",
        "Seed": SEED,
        "Test_Acc": m["test_acc"],
        "Test_F1": m["test_f1"],
        "Test_AUC": m["test_auc"],
    })
for SEED, m in results_aug.items():
    summary_rows.append({
        "Setting": "Orig+NLPaug",
        "Seed": SEED,
        "Test_Acc": m["test_acc"],
        "Test_F1": m["test_f1"],
        "Test_AUC": m["test_auc"],
    })

bert_summary_df = pd.DataFrame(summary_rows).round(4)
print(bert_summary_df)

bert_summary_df.to_csv("/content/drive/MyDrive/bert_results/bert_summary_df.csv", index=False)


In [ ]:
plot_rows = []

# LR/RF: from results_df
for _, row in results_df.iterrows():
    if row["Augmentation"] in [
        "Original", "EDA", "NLPaug", "Back-translation",
        "Original + EDA", "Original + NLPaug", "Original + Back-translation"
    ]:
        plot_rows.append({
            "Model": row["Model"],
            "Augmentation": row["Augmentation"],
            "Accuracy": row["Accuracy"],
            "CI_Lower": row["Accuracy CI Lower"],
            "CI_Upper": row["Accuracy CI Upper"],
            "Source": "Classical",
        })

# BERT: use mean over seeds for Original and Orig+NLPaug
for setting in ["Original", "Orig+NLPaug"]:
    if setting == "Original":
        res = results_orig
        aug_label = "Original"
    else:
        res = results_aug
        aug_label = "Original + NLPaug"

    accs = [m["test_acc"] for m in res.values()]
    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs, ddof=1)) if len(accs) > 1 else 0.0

    plot_rows.append({
        "Model": "BERT",
        "Augmentation": aug_label,
        "Accuracy": mean_acc,
        "CI_Lower": None,      # no CI, we’ll use std over seeds instead
        "CI_Upper": None,
        "Std_Seeds": std_acc,
        "Source": "BERT",
    })

plot_df = pd.DataFrame(plot_rows).round(4)
print(plot_df)

plot_df.to_csv("/content/drive/MyDrive/bert_results/plot_df.csv", index=False)


In [ ]:
conditions = [
    "Original",
    "EDA",
    "NLPaug",
    "Back-translation",
    "Original + EDA",
    "Original + NLPaug",
    "Original + Back-translation",
]

models = ["Logistic Regression", "Random Forest", "BERT"]
colors = {
    "Logistic Regression": "#1f77b4",
    "Random Forest": "#ff7f0e",
    "BERT": "#2ca02c",
}

x = np.arange(len(conditions))
width = 0.22

fig, ax = plt.subplots(figsize=(10, 5))

for i, model in enumerate(models):
    accs = []
    errs = []

    for cond in conditions:
        row = plot_df[(plot_df["Model"] == model) & (plot_df["Augmentation"] == cond)]
        if row.empty:
            accs.append(np.nan)
            errs.append(0.0)
        else:
            acc = float(row["Accuracy"].values[0])
            accs.append(acc)
            if model == "BERT":
                err = float(row["Std_Seeds"].values[0])
            else:
                ci_low = float(row["CI_Lower"].values[0])
                ci_up = float(row["CI_Upper"].values[0])
                err = acc - ci_low
            errs.append(err)

    offsets = x + (i - 1) * width
    ax.bar(offsets, accs, width, label=model, color=colors[model], yerr=errs, capsize=3)

ax.set_xticks(x)
ax.set_xticklabels(conditions, rotation=30, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Overall accuracy by model and augmentation")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# Save figure
fig_path = "/content/drive/MyDrive/bert_results/overall_accuracy_augmentation.png"
fig.savefig(fig_path, dpi=300)
print("Saved summarizing figure to:", fig_path)


In [ ]:
conditions = [
    "Original",
    "EDA",
    "NLPaug",
    "Back-translation",
    "Original + EDA",
    "Original + NLPaug",
    "Original + Back-translation",
]

models = ["Logistic Regression", "Random Forest", "BERT"]
markers = {
    "Logistic Regression": "o",
    "Random Forest": "s",
    "BERT": "D",
}
colors = {
    "Logistic Regression": "#1f77b4",
    "Random Forest": "#ff7f0e",
    "BERT": "#2ca02c",
}

x = np.arange(len(conditions))
offset = 0.15  # horizontal shift

fig, ax = plt.subplots(figsize=(10, 5))

for i, model in enumerate(models):
    xs = []
    ys = []
    errs = []

    for j, cond in enumerate(conditions):
        row = plot_df[(plot_df["Model"] == model) & (plot_df["Augmentation"] == cond)]
        if row.empty:
            continue
        acc = float(row["Accuracy"].values[0])
        if model == "BERT":
            err = float(row["Std_Seeds"].values[0])
        else:
            ci_low = float(row["CI_Lower"].values[0])
            ci_up = float(row["CI_Upper"].values[0])
            err = acc - ci_low  # half CI

        xs.append(j + (i - 1) * offset)
        ys.append(acc)
        errs.append(err)

    ax.errorbar(
        xs,
        ys,
        yerr=errs,
        fmt=markers[model] + "-",
        color=colors[model],
        capsize=3,
        label=model,
        linewidth=1.5,
        markersize=5,
    )

ax.set_xticks(x)
ax.set_xticklabels(conditions, rotation=30, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0.7, 1.0)  # adjust if needed
ax.set_title("Accuracy of baseline and augmented models on IMDB (5k samples)")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
fig_path = "/content/drive/MyDrive/bert_results/overall_accuracy_augmentation_lines.png"
fig.savefig(fig_path, dpi=300)
print("Saved summarizing figure to:", fig_path)


In [ ]:
# 1) Classical models: reshape for metrics
rows = []
for _, row in results_df.iterrows():
    aug = row["Augmentation"]
    model = row["Model"]
    rows.append({
        "Model": model,
        "Augmentation": aug,
        "Metric": "Accuracy",
        "Mean": row["Accuracy"],
        "Lower": row["Accuracy CI Lower"],
        "Upper": row["Accuracy CI Upper"],
    })
    rows.append({
        "Model": model,
        "Augmentation": aug,
        "Metric": "F1",
        "Mean": row["F1"],
        "Lower": row["F1 CI Lower"],
        "Upper": row["F1 CI Upper"],
    })
    if not pd.isna(row["AUC"]):
        rows.append({
            "Model": model,
            "Augmentation": aug,
            "Metric": "AUC",
            "Mean": row["AUC"],
            "Lower": row["AUC CI Lower"],
            "Upper": row["AUC CI Upper"],
        })

metrics_df = pd.DataFrame(rows)

# 2) Add BERT metrics (Original, Original+NLPaug), mean over seeds
bert_rows = []
for setting in ["Original", "Orig+NLPaug"]:
    if setting == "Original":
        res = results_orig
        aug_label = "Original"
    else:
        res = results_aug
        aug_label = "Original + NLPaug"

    accs = [m["test_acc"] for m in res.values()]
    f1s  = [m["test_f1"] for m in res.values()]
    aucs = [m["test_auc"] for m in res.values()]

    for metric_name, vals in [("Accuracy", accs), ("F1", f1s), ("AUC", aucs)]:
        mean = float(np.mean(vals))
        std  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        bert_rows.append({
            "Model": "BERT",
            "Augmentation": aug_label,
            "Metric": metric_name,
            "Mean": mean,
            "Lower": mean - std,   # use std as symmetric error
            "Upper": mean + std,
        })

metrics_df = pd.concat([metrics_df, pd.DataFrame(bert_rows)], ignore_index=True)
metrics_df = metrics_df.round(4)
metrics_df.to_csv("/content/drive/MyDrive/bert_results/metrics_df.csv", index=False)
print(metrics_df.head())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# metrics_df = pd.read_csv("/content/drive/MyDrive/bert_results/metrics_df.csv")

conditions = [
    "Original",
    "EDA",
    "NLPaug",
    "Back-translation",
    "Original + EDA",
    "Original + NLPaug",
    "Original + Back-translation",
]
metrics = ["Accuracy", "F1", "AUC"]
models = ["Logistic Regression", "Random Forest", "BERT"]
markers = {"Logistic Regression": "o", "Random Forest": "s", "BERT": "D"}
colors  = {"Logistic Regression": "#1f77b4", "Random Forest": "#ff7f0e", "BERT": "#2ca02c"}

x_positions = np.arange(len(conditions))
offset = 0.12

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=False)

for ax, metric in zip(axes, metrics):
    for i, model in enumerate(models):
        xs, ys, yerr = [], [], []
        for j, cond in enumerate(conditions):
            row = metrics_df[
                (metrics_df["Metric"] == metric) &
                (metrics_df["Model"] == model) &
                (metrics_df["Augmentation"] == cond)
            ]
            if row.empty:
                continue
            mean = float(row["Mean"].values[0])
            low  = float(row["Lower"].values[0])
            high = float(row["Upper"].values[0])
            xs.append(j + (i - 1) * offset)
            ys.append(mean)
            yerr.append(max(mean - low, high - mean))

        if xs:
            ax.errorbar(
                xs,
                ys,
                yerr=yerr,
                fmt=markers[model] + "-",
                color=colors[model],
                capsize=3,
                linewidth=1.5,
                markersize=5,
                label=model if metric == "Accuracy" else None,
            )

    ax.set_title(metric)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(conditions, rotation=45, ha="right")

axes[0].set_ylabel("Score")
fig.suptitle("Impact of data augmentation on LR, RF, and BERT (IMDB 5k)", y=1.05, fontsize=11)
axes[0].legend(frameon=False, loc="lower left")

plt.tight_layout()
fig_path = "/content/drive/MyDrive/bert_results/overall_multi_metric.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print("Saved multi-metric figure to:", fig_path)


In [ ]:
print("\n=== Combined plots for ALL seeds (Original vs Orig+NLPaug) ===")

colors = {
    12: 'tab:blue',
    44: 'tab:orange',
    127: 'tab:green',
}
linestyles_orig = '--'      # dashed for Original
linestyles_aug = '-'        # solid for Orig+NLPaug

# 1) Combined Loss Plot
plt.figure(figsize=(8, 5))
for SEED in seed_list:
    orig_m = results_orig[SEED]
    aug_m = results_aug[SEED]
    epochs_orig = range(1, len(orig_m["val_losses"]) + 1)
    epochs_aug = range(1, len(aug_m["val_losses"]) + 1)

    plt.plot(epochs_orig, orig_m["val_losses"],
             linestyle='--', color=colors[SEED],
             linewidth=2, label=f"Val Loss (Orig, seed {SEED})")
    plt.plot(epochs_aug, aug_m["val_losses"],
             linestyle='-', color=colors[SEED],
             linewidth=2, label=f"Val Loss (Orig+NLPaug, seed {SEED})")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("BERT – Validation Loss (Original vs Orig+NLPaug) across seeds")
plt.legend(ncol=2, fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("bert_val_loss_all_seeds.png", dpi=300)
plt.show()

# 2) Combined Accuracy Plot
plt.figure(figsize=(8, 5))
for SEED in seed_list:
    orig_m = results_orig[SEED]
    aug_m = results_aug[SEED]
    epochs_orig = range(1, len(orig_m["acc_epochs"]) + 1)
    epochs_aug = range(1, len(aug_m["acc_epochs"]) + 1)

    plt.plot(epochs_orig, orig_m["acc_epochs"],
             linestyle='--', color=colors[SEED],
             linewidth=2, label=f"Val Acc (Orig, seed {SEED})")
    plt.plot(epochs_aug, aug_m["acc_epochs"],
             linestyle='-', color=colors[SEED],
             linewidth=2, label=f"Val Acc (Orig+NLPaug, seed {SEED})")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("BERT – Validation Accuracy (Original vs Orig+NLPaug) across seeds")
plt.legend(ncol=2, fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("bert_val_acc_all_seeds.png", dpi=300)
plt.show()

# 3) Combined AUC Plot
plt.figure(figsize=(8, 5))
for SEED in seed_list:
    orig_m = results_orig[SEED]
    aug_m = results_aug[SEED]
    epochs_orig = range(1, len(orig_m["auc_epochs"]) + 1)
    epochs_aug = range(1, len(aug_m["auc_epochs"]) + 1)

    plt.plot(epochs_orig, orig_m["auc_epochs"],
             linestyle='--', color=colors[SEED],
             linewidth=2, label=f"Val AUC (Orig, seed {SEED})")
    plt.plot(epochs_aug, aug_m["auc_epochs"],
             linestyle='-', color=colors[SEED],
             linewidth=2, label=f"Val AUC (Orig+NLPaug, seed {SEED})")

plt.xlabel("Epoch")
plt.ylabel("Validation AUC")
plt.title("BERT – Validation AUC (Original vs Orig+NLPaug) across seeds")
plt.legend(ncol=2, fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("bert_val_auc_all_seeds.png", dpi=300)
plt.show()
